# Flux
[Flux](https://flux-framework.readthedocs.io) is a modern HPC resource manager and job scheduler similar to SLURM, but designed for hierarchical scheduling and many-task workflows. 

The basic user interaction is very similar:
| SLURM | Flux | Purpose |
|---|---|---|
| `sbatch` | `flux submit` | submit a batch job |
| `srun` | `flux run` | launch tasks interactively |
| `squeue` | `flux jobs` | inspect jobs |

Read more about flux: 
* [Integrate flux with SLURM](https://flux-framework.readthedocs.io/projects/flux-core/en/stable/guide/start.html#starting-with-slurm)
* [Cheat sheet](https://flux-framework.org/cheat-sheet/)
* [Flux Learning Guide](https://flux-framework.readthedocs.io/en/latest/guides/learning_guide.html)

## Basic Flux Commands
### `flux start`
`flux start` launches a new Flux scheduler instance inside an existing allocation.

This is commonly used together with SLURM:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun flux start workflow.sh
```

And in the workflow script `workflow.sh` we start the individual tasks as before: 
```bash
flux run -n 4 python script.py &
flux run -n 4 python script.py &
wait
```

In the jupyter environment the following commands can be directly executed.

### `flux resource list`

Inspect the available computing resources:
```bash
flux resource list
```

Example output:
```
     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-fiekunpl
 allocated      0      0     0 
      down      0      0     0 
```

In [1]:
!flux resource list

     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-5onapho1
 allocated      0      0     0 
      down      0      0     0 


### `flux jobs`

Inspect the Flux queue.
```bash
flux jobs
```
Show all jobs including completed jobs:
```bash
flux jobs -a
```

In [2]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


### `flux submit`
Submit a non-interactive job to the Flux scheduler.
```bash
flux submit --nodes=1 --ntasks=4 python script.py
```
Returns a Flux job ID.

In [3]:
lines = """\
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print(rank, size)
"""

with open("script.py", "w") as f:
    f.writelines(lines)

In [4]:
!flux submit --nodes=1 --ntasks=4 python script.py

ƒ6WHUL4F


In [5]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ6WHUL4F jovyan   python     CD      4      1   2.024s jupyter-pyiron-workshop-torlib-tutorial-5onapho1


### `flux run`
Launch a blocking interactive job.
```bash
flux run -n 4 python script.py
```
Equivalent in spirit to interactive srun.

In [6]:
!flux run -o pmi=pmix -n 4 python script.py

3 4
1 4
2 4
0 4


In [7]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒDsVFNuM jovyan   python     CD      4      1   1.897s jupyter-pyiron-workshop-torlib-tutorial-5onapho1
    ƒ6WHUL4F jovyan   python     CD      4      1   2.024s jupyter-pyiron-workshop-torlib-tutorial-5onapho1


## mpi4py Examples with Flux
### Multiple flux run Calls
In analogy to the SLURM example:
Save as `multi_flux.sh`:
```bash
#!/bin/bash
cat > task_a.py <<'PY'
from mpi4py import MPI
comm = MPI.COMM_WORLD
print(f"TASK A | rank {comm.Get_rank()} of {comm.Get_size()}", flush=True)
PY
cat > task_b.py <<'PY'
from mpi4py import MPI
comm = MPI.COMM_WORLD
print(f"TASK B | rank {comm.Get_rank()} of {comm.Get_size()}", flush=True)
PY
flux run -n 4 python task_a.py &
flux run -n 4 python task_b.py &
wait
```
Run with:
```bash
flux run -N 1 -n 8 bash multi_flux.sh
```
This launches two independent MPI jobs of size 4 each.

In [8]:
flux_script = """\
flux run -o pmi=pmix -n 4 python script.py &
flux run -o pmi=pmix -n 4 python script.py &
wait
"""
with open("flux.sh", "w") as f:
    f.writelines(flux_script)

In [10]:
!flux submit --nodes=1 --ntasks=8 bash flux.sh

ƒ3oVHTaFZ



### One flux run and Split the MPI Communicator
Save as `split_comm_flux.sh`:
```bash
#!/bin/bash
cat > workflow.py <<'PY'
from mpi4py import MPI
world = MPI.COMM_WORLD
rank = world.Get_rank()
size = world.Get_size()
color = 0 if rank < size // 2 else 1
subcomm = world.Split(color=color, key=rank)
label = "TASK A" if color == 0 else "TASK B"
print(
    f"{label} | world rank {rank} of {size} | "
    f"subrank {subcomm.Get_rank()} of {subcomm.Get_size()}",
    flush=True,
)
PY
flux run -n 8 python workflow.py
```
Run with:
```bash
bash split_comm_flux.sh
```
This launches one MPI job of size 8 and divides it into two logical communicators of size 4.

## Flux Python Interface
```python
import flux.job

jobid = flux.job.submit(
    flux.Flux(),
    jobspec=flux.job.JobspecV1.from_command(
        command=["hostname"],
        num_tasks=4,
    ),
)

print(jobid)
```